# v8 — LLM hakem: gri bölgeyi büyük LLM ile yeniden etiketle

**Tanı:** v1→v6 tüm modeller aynı etiket soyundan öğreniyor; model sınıfı atlaması +0.004, pseudo döngüsü −0.003, ensemble +0.000 verdi → tavan **etiket**, model değil. Gri bölgede (~%7, ~250K çift) tüm modeller aynı hatayı yapıyor.

**Yöntem:** Qwen2.5-32B (AWQ, vLLM ile yerel — API/ücret yok) gri bölge çiftlerini doğrudan yargılar; kararlar %25 sabit oranla birleştirilir.

## Gerekenler
1. v6 bitmiş olmalı: Drive'da `WORK/artifacts/ce_v6_test.npy` duruyor olmalı.
2. **A100** runtime (Runtime → Change runtime type). L4'te 14B modele düş (1. hücrede `LLM` değişkeni).
3. `WORK` yolunu v6'da kullandığınla AYNI yap (1. hücre).

## Süre (A100)
vLLM kurulum ~5 dk, model indirme ~5 dk, yargılama ~1.5–3 saat (250K çift, **resumable** — oturum koparsa hücreleri baştan çalıştır, kaldığı chunk'tan sürer).

## Akış
Hücreleri sırayla çalıştır. Önce `--limit 120` göz kontrolü: LLM'in örnek kararlarını oku, saçmalıyorsa devam etme.

In [ ]:
# 0) Parametreler — WORK'u v6'da kullandiginla AYNI yap!
COMPETITION = "trendyol-e-ticaret-yarismasi-2026-kaggle"
REPO_URL = "https://github.com/emrehasilik/trendyol_kaggle.git"
GITHUB_TOKEN = ""
WORK = "/content/drive/MyDrive/trendyol_kaggle"
LLM = "Qwen/Qwen2.5-32B-Instruct-AWQ"  # L4/hiz icin: "Qwen/Qwen2.5-14B-Instruct-AWQ"

In [ ]:
# 1) GPU kontrol
!nvidia-smi -L

In [ ]:
# 2) Drive + v6 skorlarinin varligini dogrula
from google.colab import drive
drive.mount("/content/drive")
import os
p = f"{WORK}/artifacts/ce_v6_test.npy"
assert os.path.exists(p), f"EKSIK: {p} — WORK yolunu v6'daki ile ayni yap (0. hucre)"
print("OK:", p)

In [ ]:
# 3) Repo
import os
url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if not os.path.exists("/content/repo"):
    !git clone -q {url} /content/repo
else:
    !cd /content/repo && git pull -q
print("repo hazir")

In [ ]:
# 4) Ortam degiskenleri
import os
os.environ["TK2_WORK"] = WORK
os.environ["TK2_LLM"] = LLM
os.environ["TK2_HF_CACHE"] = "/content/hf_cache"
print("WORK =", WORK, "| LLM =", LLM)

In [ ]:
# 5) Yarisma verisi (Kaggle API)
import os
DATA = "/content/repo/data"
os.makedirs(DATA, exist_ok=True)
if not os.path.exists(f"{DATA}/items.csv"):
    os.makedirs("/root/.kaggle", exist_ok=True)
    !cp {WORK}/kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    !kaggle competitions download -c {COMPETITION} -p {DATA}
    !cd {DATA} && unzip -o -q "*.zip" && rm -f *.zip
!ls -lh {DATA}

In [ ]:
# 6) vLLM kurulumu (~5-10 dk): surucuye uygun vllm+torch + eksik nvrtc onarimi.
#    Ara uyarilar onemsiz; SONDA 'vllm LLM import OK' gormelisin.
%pip install -q uv
import glob, os, re, subprocess
_o = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
_cuda = float(re.search(r"CUDA Version:\s*([\d.]+)", _o).group(1))
print("surucu CUDA:", _cuda)
if _cuda >= 13:
    !uv pip install --system -q --reinstall vllm --torch-backend=auto
    # libnvrtc.so.13: paketi kur, dosyayi bul, sistem yoluna bagla
    def _find():
        return glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/**/libnvrtc.so.13*",
                         recursive=True)
    _h = _find()
    if not _h:
        !uv pip install --system -q nvidia-cuda-nvrtc
        _h = _find()
    print("libnvrtc:", _h)
    assert _h, "libnvrtc.so.13 hala yok — bu ciktiyi bildir"
    _lib = sorted(_h, key=len)[-1]
    !ln -sf {_lib} /usr/lib/x86_64-linux-gnu/libnvrtc.so.13
    !ldconfig
    os.environ["LD_LIBRARY_PATH"] = (os.path.dirname(_lib) + ":" +
                                     os.environ.get("LD_LIBRARY_PATH", ""))
else:
    !uv pip install --system -q --reinstall vllm==0.11.0 --torch-backend=cu128
# gercek test: LLM sinifinin importu tum agir zinciri yukler
!python -c "from vllm import LLM; print('vllm LLM import OK')"

In [ ]:
# 7) GOZ KONTROLU: 120 ornek yargila, dosya yazmaz (~10 dk, model indirme dahil)
# Ciktida ce=CE skoru, llm=LLM karari. Ornekleri OKU: kararlar mantikliysa devam.
!cd /content/repo/src && python -u llm_judge_v8.py judge --tag v6 --limit 120

In [ ]:
# 8) TAM KOSU: gri bolgeyi yargila (resumable; kopursa bu hucreyi tekrar calistir)
!cd /content/repo/src && python -u llm_judge_v8.py judge --tag v6

In [ ]:
# 9) Birlestir + submission (GPU gerekmez, saniyeler surer)
# alpha = LLM'e guven agirligi (gri bolgede). Iki dosya uret:
!cd /content/repo/src && python -u llm_judge_v8.py merge --tag v6 --alpha 0.7
!cd /content/repo/src && python -u llm_judge_v8.py merge --tag v6 --alpha 1.0

In [ ]:
# 10) Gonder: ONCE alpha 0.7 dosyasini. Yukselirse alpha 1.0'i da dene.
print("dosyalar:")
!ls -lh {WORK}/output/sub_v6_llm*.csv
# !kaggle competitions submit -c {COMPETITION} -f {WORK}/output/sub_v6_llm70_rate25.csv -m "v6 + LLM hakem alpha0.7 rate25"

## Tur 2 + v9 — LLM bilgisini modele taşı

**Sonuçlar:** `llm70` = 0.860, `llm100` = 0.861 (rekor). alpha 1.0 > 0.7 → gri bölgede LLM, CE'den iyi; LLM'e tam güven doğru.

**Sırada iki hamle (sırayla):**
- **A) Tur 2:** bandı genişlet (0.02–0.98), sadece YENİ çiftleri yargıla (~30–60 dk) → `sub_v6_llm100_rate25_r2.csv`
- **B) v9:** LLM etiketlerini eğitim setine kat, CE'yi yeniden eğit (~4–6 saat, resumable) → model düzeltmeleri tüm 3.36M çifte genellestirir → `sub_v9_rate25.csv` ve `sub_v9_llm100_rate25.csv`

Oturum koparsa: 0–6. hücreleri çalıştır (kurulum), sonra buradan devam — her adım resumable.

In [ ]:
# 11) TUR 2: bandi genislet, yalniz YENI ciftleri yargila (resumable)
!cd /content/repo && git pull -q
!cd /content/repo/src && python -u llm_judge_v8.py judge --tag v6 --name v6r2 --lo 0.02 --hi 0.98 --exclude v6

In [ ]:
# 12) Tur 1+2 birlesik submission -> sub_v6_llm100_rate25_r2.csv (GONDER)
!cd /content/repo/src && python -u llm_judge_v8.py merge --tag v6 --names v6,v6r2 --alpha 1.0 --suffix r2
# !kaggle competitions submit -c {COMPETITION} -f {WORK}/output/sub_v6_llm100_rate25_r2.csv -m "v6 + LLM hakem tur2 alpha1.0 rate25"

In [ ]:
# 13) v9 dataseti (v5 tabani + LLM etiketleri x3) + egitim (~4-6 saat, resumable)
# Egitim ciktisinda ilk 50 adimda [smoke] satirini kontrol et (loss dusuyor mu).
%pip install -q scikit-learn pyarrow
!cd /content/repo/src && python -u build_dataset_v9.py --names v6,v6r2
!cd /content/repo/src && python -u train_ce_v6.py --data train_dataset_v9.parquet --tag v9

In [ ]:
# 14) v9 test skorlama + submission -> sub_v9_rate25.csv (GONDER: retrain etkisi izole)
!cd /content/repo/src && python -u infer_ce_v6.py --tag v9 --rate 0.25
# !kaggle competitions submit -c {COMPETITION} -f {WORK}/output/sub_v9_rate25.csv -m "v9 LLM-etiketli retrain rate25"

In [ ]:
# 15) v9'un YENI gri bolgesini yargila + tum turlari birlestir
#     -> sub_v9_llm100_rate25.csv (GONDER: ana aday)
!cd /content/repo/src && python -u llm_judge_v8.py judge --tag v9 --name v9 --lo 0.02 --hi 0.98 --exclude v6,v6r2
!cd /content/repo/src && python -u llm_judge_v8.py merge --tag v9 --names v6,v6r2,v9 --alpha 1.0
# !kaggle competitions submit -c {COMPETITION} -f {WORK}/output/sub_v9_llm100_rate25.csv -m "v9 + LLM hakem alpha1.0 rate25"